# SamplerV2 and finite-shot counts

Compare Qiskit's StatevectorSampler with MettleQSamplerV2 using the same Bell-state sampling contract.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.measure_all()
shots = 4096

def run_reference():
    result = StatevectorSampler(seed=19).run([circuit], shots=shots).result()[0]
    return result.data.meas.get_counts()

reference, reference_ms, _ = benchmark(run_reference)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    result = MettleQSamplerV2(backend=backend).run([compiled], shots=shots).result()[0]
    return result.data.meas.get_counts()

candidate, mettleq_ms, _ = benchmark(run_mettleq)
tvd = total_variation_distance(reference, candidate)
support_ok = set(reference) <= {"000", "111"} and set(candidate) <= {"000", "111"}
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/03_sampler_and_counts.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="finite-shot total-variation distance <= 0.05",
    passed=support_ok and tvd <= 0.05,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "reference_counts": reference, "mettleq_counts": candidate},
    notes="Independent sampler RNGs are compared statistically, not byte-for-byte.",
)

TUTORIAL_RESULT::{"check": "finite-shot total-variation distance <= 0.05", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"mettleq_counts": {"000": 2104, "111": 1992}, "reference_counts": {"000": 2053, "111": 2043}, "tvd": 0.012451171875}, "mettleq_median_ms": 6.793832988478243, "notebook": "qiskit/03_sampler_and_counts.ipynb", "notes": "Independent sampler RNGs are compared statistically, not byte-for-byte.", "passed": true, "python": "3.13.2", "reference_median_ms": 4.4669169874396175, "reference_over_mettleq": 0.6574958488109914, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
